In [4]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd
import plotly.express as px

uncertainty = pd.read_csv('data/processed/uncertainty_metrics.csv')
inventory_risk = pd.read_csv('data/processed/inventory_risk.csv')

print(uncertainty[['StockCode', 'DataQualityFlag', 'MeanRelativeUncertainty']])

   StockCode                        DataQualityFlag  MeanRelativeUncertainty
0      23843  Low reliability - sparse/spike-driven                   1371.2
1      23166  Low reliability - sparse/spike-driven                   1056.7
2      21915                                     OK                    459.7
3      22086                                     OK                    431.6
4      22197                                     OK                    370.0
5      21977                                     OK                    345.5
6      15036                                     OK                    331.4
7      84077                                     OK                    317.6
8      84879                                     OK                    317.3
9      17003                                     OK                    314.6
10     23203                                     OK                    299.6
11     22469                                     OK                    294.7

In [5]:
fig = px.bar(
    uncertainty,
    x='StockCode',
    y='MeanRelativeUncertainty',
    color='DataQualityFlag',
    color_discrete_map={
        'OK': '#1f77b4',
        'Low reliability - sparse/spike-driven': '#d62728'
    },
    title='Forecast Uncertainty by Product (% of historical mean)',
    labels={'MeanRelativeUncertainty': 'Relative Uncertainty (%)'}
)
fig.show()

In [6]:
risk_counts = inventory_risk['RiskStatus'].value_counts().reset_index()
fig2 = px.pie(
    risk_counts,
    names='RiskStatus',
    values='count',
    title='Inventory Risk Status Distribution',
    color='RiskStatus',
    color_discrete_map={
        'Stockout Risk': 'red',
        'Low Stock': 'orange',
        'Adequate': 'green',
        'Overstock Risk': 'purple',
        'Unreliable Forecast': 'gray'
    }
)
fig2.show()

print(inventory_risk[['StockCode', 'Description', 
                        'CurrentStock', 'ForecastedDemand',
                        'RiskStatus', 'RiskReason']].to_string(index=False))

StockCode                        Description  CurrentStock  ForecastedDemand          RiskStatus                                                     RiskReason
    23843        PAPER CRAFT , LITTLE BIRDIE   7864.004087      24689.088394 Unreliable Forecast Data too sparse/spike-driven for a trustworthy demand estimate
    22197                     POPCORN HOLDER   3384.579145       6739.630442           Low Stock                             May run short if demand is average
   85099B            JUMBO BAG RED RETROSPOT   3113.983688       4404.930377           Low Stock                             May run short if demand is average
    84077  WORLD WAR 2 GLIDERS ASSTD DESIGNS   3644.466401       4003.955492           Low Stock                             May run short if demand is average
    84879      ASSORTED COLOUR BIRD ORNAMENT   2425.559004       3700.143757           Low Stock                             May run short if demand is average
    23203           JUMBO BAG VINTAGE DO

## Day 3 — Notes & Observations

### What each cell did

**Data loading cell** — loaded `uncertainty_metrics.csv` and `inventory_risk.csv`, confirming both correctly carry the `DataQualityFlag` from earlier pipeline stages.

**Uncertainty bar chart** — plotted `MeanRelativeUncertainty` per product, colored by data quality flag. The two flagged products (23843: 1371.2%, 23166: 1056.7%) are clearly separated from all 18 "OK" products, which cluster in a tighter 210-460% band. This confirms the data-quality flag and the independently-calculated uncertainty metric agree - two different signals pointing at the same two products.

**Inventory risk pie chart** — final RiskStatus distribution: 60% Low Stock, 30% Adequate, 10% Unreliable Forecast, 0% Stockout Risk.

### Fix 1: uncertainty metric hidden by clipping

`calculate_uncertainty_metrics` originally divided IntervalWidth by `yhat` (the point forecast). Product 23166's `yhat` is clipped to exactly 0 across all future weeks (its raw Prophet forecast was negative - see Day 2 notes), which made its relative uncertainty compute as a misleading 0.0% - the opposite of the truth, since this product has one of the least trustworthy forecasts in the dataset. Fixed by dividing by each product's historical mean demand instead (a value that's never zero for a real product), which correctly moved 23166 to the second-highest 
uncertainty ranking (1056.7%).

### Fix 2: inventory risk misclassifying unreliable products

The same root issue affected `calculate_inventory_risk`: 23166 showed as "Adequate" (0 forecasted demand vs. current stock - technically correct given clipped-to-zero yhat, but misleading in practice, since this product's demand isn't actually zero, just unforecastable). Fixed by adding an explicit check: any product with `DataQualityFlag != 'OK'` is labeled "Unreliable Forecast" rather than receiving a computed risk status.

### Fix 3: stock/demand horizon mismatch

Initially 80% of products showed "Low Stock." Part of this was a real bug: the synthetic CurrentStock assumption used a 3-week window while ForecastedDemand summed 4 weeks - a built-in mismatch that structurally inflated the Low Stock count regardless of actual demand patterns. Fixed by aligning both to 4 weeks, which moved the distribution to 60% Low Stock / 30% Adequate / 10% Unreliable Forecast / 0% Stockout Risk.

The remaining 60% Low Stock is a more genuine pattern, not a bug: CurrentStock is estimated from *historical average* demand, while  forecastedDemand comes from each product's fitted *trend*, and many products show a Growing trend (see Day 2 analysis). A backward-looking average will naturally undershoot a forward-projecting upward trend, which is why most products still show as understocked even after the horizon fix.

Note: after the 4-week alignment, no products remain classified as "Stockout Risk" (23203 moved into "Low Stock" once its stock assumption grew from 3 to 4 weeks of average demand). This is an expected mechanical consequence of raising the stock assumption, not a new finding - it further underscores that RiskStatus values here depend heavily on the synthetic CurrentStock assumption rather than real inventory data.

### What remains a genuine limitation (not fixable in code)

CurrentStock is entirely synthetic throughout this analysis - the UCI dataset has no real inventory-level data. The 4-week alignment removes a self-inflicted inconsistency, but it does not make the underlying stock figures real. This is a limitation of the data source, not the model, and belongs in the README's limitations section rather than something further code changes can resolve.

### Why wide confidence intervals matter for inventory decisions

A narrow interval (e.g. 84946 at 210%) means Prophet is relatively confident in its estimate - safer to plan around the point forecast alone. A wide interval (e.g. 21915 at 460%, and far more so 23843/23166) means true demand could reasonably fall anywhere across a huge range - planning inventory around just the point forecast for these products would be risky. This is exactly why the Unreliable Forecast label matters: for the two flagged products, the interval is so wide relative to the forecast that the point estimate alone is close to meaningless for a real stocking decision.